Cloud-only data dependency: this notebook expects access to OncDRS/cloud data paths and Application Default Credentials with permission to call Vertex AI.

# Binary NEPC Pipeline Runner — Vertex AI

This notebook mirrors the GPT-based `binary_NEPC/generate_notes_and_run_llm.ipynb` workflow while running the classifier through Vertex AI in project `gusevlabllm`.

Pipeline steps:
1. Run `shared/compile_prostate_notes.py` to build `prostate_text_data.csv`.
2. Optionally run `binary_NEPC/compile_prostate_note_bundle.py` to build a gzip note bundle.
3. Run `binary_NEPC/compile_patient_snippets.py` to save the patient-level trigger snippets.
4. Run `binary_NEPC/run_NEPC_classifier.py` against that saved artifact, with parallel Vertex AI requests controlled by `MAX_WORKERS`.

All run toggles default to `False`; review the printed commands and paths before enabling a step.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys


def find_vertex_root(start):
    """Find the vertex_ai source root from the repo root or notebook directory."""
    start = Path(start).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if candidate.name == "vertex_ai" and (candidate / "binary_NEPC").is_dir():
            return candidate
        nested = candidate / "vertex_ai"
        if (nested / "binary_NEPC").is_dir():
            return nested
    raise FileNotFoundError("Could not locate the vertex_ai source directory")


VERTEX_ROOT = find_vertex_root(Path.cwd())
PYTHON = sys.executable


def shell_join(parts):
    return " ".join(shlex.quote(str(part)) for part in parts)


def run_command(parts, env=None):
    print(shell_join(parts))
    completed = subprocess.run(parts, cwd=VERTEX_ROOT, env=env, check=False)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")


DEFAULT_DATA_ROOT = Path(
    os.environ.get("LLM_ANNOTATIONS_DATA_PATH", "/data/gusev/USERS/jpconnor/data/LLM_annotations/")
)
DEFAULT_NOTES_CSV = DEFAULT_DATA_ROOT / "prostate_text_data.csv"
DEFAULT_NOTE_BUNDLE = DEFAULT_DATA_ROOT / "LLM_NEPC_labels" / "LLM_NEPC_classifier_note_bundle.json.gz"
DEFAULT_SNIPPET_BUNDLE = DEFAULT_DATA_ROOT / "LLM_NEPC_labels" / "LLM_NEPC_classifier_patient_snippets.json.gz"
DEFAULT_LABELS_DIR = DEFAULT_DATA_ROOT / "LLM_NEPC_labels"
DEFAULT_COHORT_SOURCE = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/prostate_arpi_survival_cohort.csv")

print(f"Vertex AI source root: {VERTEX_ROOT}")

In [ ]:
# Parameters
PROJECT_ID = "gusevlabllm"
LOCATION = "us-central1"

MRNS = []
MRN_FILE = "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/mrn_lists/icd_or_vte_mrns.csv"
COHORT_SOURCE = DEFAULT_COHORT_SOURCE
NOTES_CSV_PATH = DEFAULT_NOTES_CSV
NOTE_BUNDLE_PATH = DEFAULT_NOTE_BUNDLE
SNIPPET_BUNDLE_PATH = DEFAULT_SNIPPET_BUNDLE
OUTPUT_DIR = DEFAULT_LABELS_DIR

# Repeat entries here to override the default raw OncDRS note roots.
RAW_TEXT_PATHS = []

# Snippet compilation settings
MAX_NOTES_PER_PATIENT = 75    # per-patient snippet cap
OVERWRITE_SNIPPETS = False

# Vertex AI classifier settings
MODEL_NAME = "gemini-2.5-flash-lite"
MAX_WORKERS = 4               # concurrent patient-level Vertex AI requests
MAX_RETRIES = 6
LIMIT_MRNS = None
OVERWRITE = False

# Step toggles
RUN_COMPILE_NOTES = False
RUN_COMPILE_BUNDLE = False
RUN_COMPILE_SNIPPETS = False  # required before classification
RUN_CLASSIFIER = False
RUN_RETRY_FAILURES = False    # rerun only patients in the failed/unlabeled TSV

# Pass Vertex configuration to every child process. Existing environment values
# are preserved except for these explicit notebook parameters.
RUN_ENV = os.environ.copy()
RUN_ENV["VERTEX_PROJECT"] = PROJECT_ID
RUN_ENV["VERTEX_LOCATION"] = LOCATION
RUN_ENV["VERTEX_MODEL"] = MODEL_NAME

In [ ]:
notes_csv_exists = Path(NOTES_CSV_PATH).exists()
note_bundle_exists = Path(NOTE_BUNDLE_PATH).exists()
snippet_bundle_exists = Path(SNIPPET_BUNDLE_PATH).exists()
labels_path = Path(OUTPUT_DIR) / "LLM_NEPC_classifier_labels.tsv"
labels_exist = labels_path.exists()

print(f"Vertex project: {PROJECT_ID}")
print(f"Vertex location: {LOCATION}")
print(f"Vertex model: {MODEL_NAME}")
print(f"Parallel workers: {MAX_WORKERS}")
print(f"Compiled notes CSV exists: {notes_csv_exists}")
print(f"Path: {NOTES_CSV_PATH}")
print(f"Note bundle exists: {note_bundle_exists}")
print(f"Path: {NOTE_BUNDLE_PATH}")
print(f"Patient snippet bundle exists: {snippet_bundle_exists}")
print(f"Path: {SNIPPET_BUNDLE_PATH}")
print(f"Labels TSV exists: {labels_exist}")
print(f"Path: {labels_path}")

In [ ]:
compile_notes_cmd = [
    PYTHON,
    "shared/compile_prostate_notes.py",
    "--output-path",
    NOTES_CSV_PATH,
    "--cohort-source",
    COHORT_SOURCE,
]

if MRNS:
    compile_notes_cmd.extend(["--mrns", ",".join(str(mrn) for mrn in MRNS)])
if MRN_FILE is not None:
    compile_notes_cmd.extend(["--mrn-file", MRN_FILE])
if RAW_TEXT_PATHS:
    for raw_text_path in RAW_TEXT_PATHS:
        compile_notes_cmd.extend(["--raw-text-path", raw_text_path])

print(shell_join(compile_notes_cmd))

In [ ]:
if RUN_COMPILE_NOTES:
    run_command(compile_notes_cmd, env=RUN_ENV)
else:
    print("Skipping compile_prostate_notes.py")

In [ ]:
compile_bundle_cmd = [
    PYTHON,
    "binary_NEPC/compile_prostate_note_bundle.py",
    "--output-path",
    NOTE_BUNDLE_PATH,
    "--notes-csv",
    NOTES_CSV_PATH,
]

if MRNS:
    compile_bundle_cmd.extend(["--mrns", ",".join(str(mrn) for mrn in MRNS)])
if MRN_FILE is not None:
    compile_bundle_cmd.extend(["--mrn-file", MRN_FILE])
if RAW_TEXT_PATHS:
    for raw_text_path in RAW_TEXT_PATHS:
        compile_bundle_cmd.extend(["--raw-text-path", raw_text_path])

print(shell_join(compile_bundle_cmd))

In [ ]:
if RUN_COMPILE_BUNDLE:
    run_command(compile_bundle_cmd, env=RUN_ENV)
else:
    print("Skipping compile_prostate_note_bundle.py")

In [ ]:
compile_snippets_cmd = [
    PYTHON,
    "binary_NEPC/compile_patient_snippets.py",
    "--notes-csv",
    NOTES_CSV_PATH,
    "--output-path",
    SNIPPET_BUNDLE_PATH,
    "--max-notes-per-patient",
    str(MAX_NOTES_PER_PATIENT),
]

if Path(NOTE_BUNDLE_PATH).exists():
    compile_snippets_cmd.extend(["--note-bundle-path", NOTE_BUNDLE_PATH])
if MRNS:
    compile_snippets_cmd.extend(["--mrns", ",".join(str(mrn) for mrn in MRNS)])
if MRN_FILE is not None:
    compile_snippets_cmd.extend(["--mrn-file", MRN_FILE])
if RAW_TEXT_PATHS:
    for raw_text_path in RAW_TEXT_PATHS:
        compile_snippets_cmd.extend(["--raw-text-path", raw_text_path])
if OVERWRITE_SNIPPETS:
    compile_snippets_cmd.append("--overwrite")

print(shell_join(compile_snippets_cmd))

In [ ]:
if RUN_COMPILE_SNIPPETS:
    run_command(compile_snippets_cmd, env=RUN_ENV)
else:
    print("Skipping compile_patient_snippets.py")

In [ ]:
run_classifier_cmd = [
    PYTHON,
    "binary_NEPC/run_NEPC_classifier.py",
    "--snippets-path",
    SNIPPET_BUNDLE_PATH,
    "--output-dir",
    OUTPUT_DIR,
    "--model",
    MODEL_NAME,
    "--max-workers",
    str(MAX_WORKERS),
    "--max-retries",
    str(MAX_RETRIES),
]

if MRNS:
    run_classifier_cmd.extend(["--mrns", ",".join(str(mrn) for mrn in MRNS)])
if MRN_FILE is not None:
    run_classifier_cmd.extend(["--mrn-file", MRN_FILE])
if LIMIT_MRNS is not None:
    run_classifier_cmd.extend(["--limit-mrns", str(LIMIT_MRNS)])
if OVERWRITE:
    run_classifier_cmd.append("--overwrite")

print(shell_join(run_classifier_cmd))

In [ ]:
if RUN_CLASSIFIER:
    run_command(run_classifier_cmd, env=RUN_ENV)
else:
    print("Skipping run_NEPC_classifier.py")

In [ ]:
# Rerun only failed/unlabeled patients. Successful retries are added to labels
# and removed from the failures file.
retry_failed_cmd = [part for part in run_classifier_cmd if part != "--overwrite"]
retry_failed_cmd.append("--retry-failures")
print(shell_join(retry_failed_cmd))

if RUN_RETRY_FAILURES:
    run_command(retry_failed_cmd, env=RUN_ENV)
else:
    print("Skipping failed-patient retry")